In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_recall_fscore_support,
)

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 1) Daten laden

In [ ]:
CSV_PATH = "free_throw_dataset.csv"
if not os.path.exists(CSV_PATH):
    alt = os.path.join("/mnt/data", CSV_PATH)
    if os.path.exists(alt):
        CSV_PATH = alt

df = pd.read_csv(CSV_PATH)
df["point_index"] = df.groupby("wurf_id").cumcount()
print("CSV_PATH:", CSV_PATH)
print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
per_throw = (
    df.groupby("wurf_id")
    .agg({"label": "first", "theta": "first", "v0": "first", "T": "first"})
    .reset_index()
)

print("Anzahl Würfe:", per_throw.shape[0])
print(per_throw["label"].value_counts())
per_throw.head()

In [ ]:
train_ids, test_ids = train_test_split(
    per_throw["wurf_id"].values,
    test_size=0.15,
    random_state=SEED,
    stratify=per_throw["label"].values,
)
train_ids, val_ids = train_test_split(
    train_ids,
    test_size=0.15,
    random_state=SEED,
    stratify=per_throw.set_index("wurf_id").loc[train_ids, "label"].values,
)

print("Train/Val/Test:", len(train_ids), len(val_ids), len(test_ids))
print("Alle Test ids:", test_ids)

df_train = (
    df[df["wurf_id"].isin(train_ids)]
    .sort_values(["wurf_id", "point_index"])
    .reset_index(drop=True)
)
df_val = (
    df[df["wurf_id"].isin(val_ids)]
    .sort_values(["wurf_id", "point_index"])
    .reset_index(drop=True)
)
df_test = (
    df[df["wurf_id"].isin(test_ids)]
    .sort_values(["wurf_id", "point_index"])
    .reset_index(drop=True)
)

## 3) Scaling (fit auf Train)

In [ ]:
USE_XY_STANDARDIZATION = False  # Vergleich: False (Meter-Raum)

class IdentityScaler:
    """No-op Scaler: transformiert nicht (Einheiten bleiben Meter).
    mean_/std_ sind definiert, damit Downstream-Code (z. B. Ringhöhe im Modellraum) konsistent bleibt.
    """
    def __init__(self):
        self.mean_ = np.zeros((1, 2), dtype=np.float32)
        self.std_  = np.ones((1, 2), dtype=np.float32)

    def fit(self, xy: np.ndarray):
        return self

    def transform(self, xy: np.ndarray) -> np.ndarray:
        return xy.astype(np.float32, copy=False)

    def inverse_transform(self, xy_scaled: np.ndarray) -> np.ndarray:
        return xy_scaled.astype(np.float32, copy=False)


class XYScaler:
    """Standardisierung (optional)."""
    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, xy: np.ndarray):
        self.mean_ = xy.mean(axis=0, keepdims=True)
        self.std_ = xy.std(axis=0, keepdims=True) + 1e-8
        return self

    def transform(self, xy: np.ndarray) -> np.ndarray:
        return (xy - self.mean_) / self.std_

    def inverse_transform(self, xy_scaled: np.ndarray) -> np.ndarray:
        return xy_scaled * self.std_ + self.mean_


train_xy = df_train[["x_clean","y_clean"]].to_numpy(np.float32)

scaler = XYScaler().fit(train_xy) if USE_XY_STANDARDIZATION else IdentityScaler().fit(train_xy)

print("USE_XY_STANDARDIZATION:", USE_XY_STANDARDIZATION)
print("scaler.mean_:", scaler.mean_)
print("scaler.std_ :", scaler.std_)


## 4) Dataset

In [ ]:
class ExtrapolationDataset(Dataset):
    def __init__(self, df, scaler, tin, input_source="clean", target_source="clean"):
        self.df = df
        self.scaler = scaler
        self.tin = tin
        self.input_source = input_source
        self.target_source = target_source

        # >>> neu: wurf_id-Liste + Gruppen, damit wir für idx wieder an die Rohdaten kommen
        self.throw_ids = list(self.df["wurf_id"].unique())
        self.groups = {k: g.sort_values("point_index").reset_index(drop=True)
                       for k, g in self.df.groupby("wurf_id")}

    def __len__(self):
        return len(self.throw_ids)

    def __getitem__(self, idx):
        wid = self.throw_ids[idx]
        g = self.groups[wid]
        label = int(g["label"].iloc[0])

        T = len(g)
        assert self.tin <= T

        
        t = g["t"].to_numpy(np.float32)[:, None]        
        t_min = float(t[0, 0])
        t_obs_max = float(t[self.tin - 1, 0])           

        denom = max(1e-8, (t_obs_max - t_min))          
        t_n = (t - t_min) / denom                       

        if self.tin >= 2:
            dt_n = float(np.mean(np.diff(t_n[: self.tin, 0])))
        else:
            dt_n = 0.0

        xy_clean = g[["x_clean","y_clean"]].to_numpy(np.float32)
        xy_noisy = g[["x_noisy","y_noisy"]].to_numpy(np.float32)

        xy_in  = xy_noisy if self.input_source=="noisy" else xy_clean
        xy_tgt = xy_clean if self.target_source=="clean" else xy_noisy

        xy_in_s  = self.scaler.transform(xy_in)    
        xy_tgt_s = self.scaler.transform(xy_tgt)   

        x_in = np.concatenate([xy_in_s[:self.tin], t_n[:self.tin]], axis=1).astype(np.float32)

        y_all = xy_tgt_s.astype(np.float32)

        return (
            torch.from_numpy(x_in),                                 
            torch.tensor([dt_n], dtype=torch.float32),              
            torch.from_numpy(y_all),                                
            torch.tensor(label, dtype=torch.long)
        )


def collate_extrap(batch):
    x_in, dt, y_all, labels = zip(*batch)
    return (
        torch.stack(x_in, dim=0),     
        torch.stack(dt, dim=0),       
        torch.stack(y_all, dim=0),    
        torch.stack(labels, dim=0),  
    )


## 5) Modelle

In [ ]:
class ExtrapSeq2SeqModel(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=128, num_layers=2, n_classes=3, dropout=0.1):
        super().__init__()
        self.enc = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=(dropout if num_layers > 1 else 0.0),
        )
        # Decoder bekommt: prev_xy (2) + dt (1) => 3
        self.dec_cell = nn.LSTMCell(input_size=3, hidden_size=hidden_dim)
        self.out_head = nn.Linear(hidden_dim, 2)
        self.cls_head = nn.Linear(hidden_dim, n_classes)

    def forward(self, x_in, dt, n_future: int = 0, return_warmup_preds: bool = False):
        out, (hN, cN) = self.enc(x_in)      # out: (B,Tin,H)
        h = hN[-1]                           # (B,H)
        c = cN[-1]

        logits = self.cls_head(h)            # (B,3)

        B, Tin, _ = x_in.shape

        # ----------------------------
        # 1) Teacher-forced Warm-up über den beobachteten Teil
        #    Optional: 1‑Step‑Predictions im Beobachtungsfenster sammeln.
        # ----------------------------
        prev_xy = x_in[:, 0, :2]
        warm_preds = [] if return_warmup_preds else None

        for k in range(1, Tin):
            dec_in = torch.cat([prev_xy, dt], dim=1)
            h, c = self.dec_cell(dec_in, (h, c))

            if return_warmup_preds:
                xy_hat = self.out_head(h)             
                warm_preds.append(xy_hat.unsqueeze(1))

            prev_xy = x_in[:, k, :2]                 

        # ----------------------------
        # 2) Autoregressiver Rollout für Zukunftsschritte (ohne Teacher Forcing)
        # ----------------------------
        preds_f = []
        for _ in range(int(n_future)):
            dec_in = torch.cat([prev_xy, dt], dim=1)
            h, c = self.dec_cell(dec_in, (h, c))
            xy_next = self.out_head(h)                
            preds_f.append(xy_next.unsqueeze(1))
            prev_xy = xy_next                         

        if int(n_future) > 0:
            y_future = torch.cat(preds_f, dim=1)      # (B,n_future,2)
            y_hat_full = torch.cat([x_in[:, :, :2], y_future], dim=1)
        else:
            y_hat_full = x_in[:, :, :2]

        if return_warmup_preds:
            if Tin <= 1:
                warmup_preds = torch.zeros((B, 0, 2), device=x_in.device, dtype=x_in.dtype)
            else:
                warmup_preds = torch.cat(warm_preds, dim=1)  # (B,Tin-1,2)
            return y_hat_full, logits, warmup_preds

        return y_hat_full, logits

    @torch.no_grad()
    def rollout_until_ring(self, x_in, dt, y_ring_scaled, max_steps=300):
        self.eval()

        B, Tin, _ = x_in.shape
        device = x_in.device
        dtype = x_in.dtype

        if torch.is_tensor(y_ring_scaled):
            y_ring = y_ring_scaled.to(device=device, dtype=dtype)
            if y_ring.dim() == 0:
                y_ring = y_ring.expand(B)
            elif y_ring.dim() == 2 and y_ring.size(1) == 1:
                y_ring = y_ring[:, 0]
            elif y_ring.dim() == 1:
                if y_ring.numel() == 1:
                    y_ring = y_ring.expand(B)
            else:
                y_ring = y_ring.reshape(B)
        else:
            y_ring = torch.full((B,), float(y_ring_scaled), device=device, dtype=dtype)

        if max_steps is None or int(max_steps) <= 0:
            y_future = torch.zeros((B, 0, 2), device=device, dtype=dtype)
            stop_idx = torch.full((B,), -1, dtype=torch.long, device=device)
            return y_future, stop_idx, 0

        out, (hN, cN) = self.enc(x_in)
        h = hN[-1]
        c = cN[-1]

        # ----------------------------
        # Warm-up im Beobachtungsfenster (teacher-forced)
        # ----------------------------
        prev_xy = x_in[:, 0, :2]        # (B,2)
        for k in range(1, Tin):
            dec_in = torch.cat([prev_xy, dt], dim=1)
            h, c = self.dec_cell(dec_in, (h, c))
            prev_xy = x_in[:, k, :2]

        # Start für Zukunftsrollout: letzter beobachteter Punkt
        prev_xy = x_in[:, -1, :2]       # (B,2)
        prev_y = prev_xy[:, 1]          # (B,)

        # Apex-Status initialisieren: wenn die letzten beobachteten Punkte bereits fallend sind -> Apex passiert
        if Tin >= 2:
            dy_obs = x_in[:, -1, 1] - x_in[:, -2, 1]  # (B,)
            passed_apex = (dy_obs < 0).to(torch.bool)
        else:
            passed_apex = torch.zeros(B, dtype=torch.bool, device=device)

        # Falls der letzte beobachtete Punkt bereits nach dem Apex auf/unter Ringhöhe liegt -> fertig
        already_done = passed_apex & (prev_y <= y_ring)  # (B,)
        if torch.all(already_done):
            y_future = torch.zeros((B, 0, 2), device=device, dtype=dtype)
            stop_idx = torch.full((B,), -1, dtype=torch.long, device=device)
            return y_future, stop_idx, 0

        preds = []
        stop_idx = torch.full((B,), fill_value=int(max_steps) - 1, dtype=torch.long, device=device)
        stopped = torch.zeros(B, dtype=torch.bool, device=device)

        for step in range(int(max_steps)):
            dec_in = torch.cat([prev_xy, dt], dim=1)
            h, c = self.dec_cell(dec_in, (h, c))
            xy = self.out_head(h)       # (B,2)
            preds.append(xy.unsqueeze(1))

            y = xy[:, 1]                # (B,)
            dy = y - prev_y             # (B,)
            passed_apex = passed_apex | (dy < 0)

            stop_now = (~stopped) & passed_apex & (y <= y_ring)  # (B,)
            if torch.any(stop_now):
                stop_idx = torch.where(stop_now, stop_idx.new_full((B,), step), stop_idx)
                stopped = stopped | stop_now

            prev_xy = xy
            prev_y = y

            if torch.all(stopped):
                break

        y_future = torch.cat(preds, dim=1) if preds else torch.zeros((B, 0, 2), device=device, dtype=dtype)
        S = int(y_future.size(1))
        if S > 0:
            stop_idx = torch.clamp(stop_idx, min=0, max=S - 1)
        return y_future, stop_idx, S

## 6) Training & Evaluation Helpers

In [ ]:
def _get_scaler_or_raise(scaler):
    if scaler is None:
        scaler = globals().get("scaler", None)
    if scaler is None:
        raise RuntimeError("Kein 'scaler' verfügbar. Übergib ihn an train_extrap/eval_extrap oder definiere ihn global.")
    return scaler


def y_ring_scaled_from_scaler(scaler, y_ring_m: float = 3.048) -> float:
    mean_y = float(scaler.mean_[0, 1])
    std_y  = float(scaler.std_[0, 1])
    return (float(y_ring_m) - mean_y) / std_y


def train_extrap(
    model: ExtrapSeq2SeqModel,
    train_loader: DataLoader,
    val_loader: DataLoader,
    tin: int = 30,
    max_steps_cap: int = 300,
    epochs: int = 800,
    lr: float = 2e-3,
    cls_weight: float | None = None,
    cls_target_ratio: float = 0.01,
    calibrate_cls_weight: bool = True,
    patience: int = 25,
    grad_accum_steps: int = 1,
    y_ring_m: float = 3.048,
    scaler=None,
    max_grad_norm: float | None = None,
    verbose: bool = True,
):
    scaler = _get_scaler_or_raise(scaler)
    _ = y_ring_scaled_from_scaler(scaler, y_ring_m=y_ring_m)  # Konsistenz (Meter->Modellraum)

    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    hist = {
        "tr_loss": [],
        "va_loss": [],
        "tr_acc": [],
        "va_acc": [],
        "tr_mse_obs": [],
        "va_mse_obs": [],
        "cls_weight": [],
    }

    best = {"loss": float("inf"), "state": None}
    bad_epochs = 0

    # cls_weight Kalibrierung (einmalig pro Training)
    cls_w = cls_weight
    calibrated = False

    for ep in range(1, epochs + 1):
        # ---------------- train ----------------
        model.train()
        tr_losses = []
        tr_y, tr_p = [], []
        tr_mse_obs = []

        opt.zero_grad(set_to_none=True)
        for step, (x_in, dt, y_all, c) in enumerate(train_loader):
            x_in = x_in.to(device)
            dt = dt.to(device)
            c = c.to(device)

            Tin = int(x_in.size(1))

            # Strikt: kein Future-Unroll im Training
            yhat, logits, warm_preds = model(x_in, dt, n_future=0, return_warmup_preds=True)

            if Tin > 1:
                tgt_obs = x_in[:, 1:Tin, :2]                 # (B,Tin-1,2)
                loss_obs = F.mse_loss(warm_preds, tgt_obs)   # 1-step loss innerhalb obs
                mse_obs = float(loss_obs.detach().item())
            else:
                loss_obs = torch.tensor(0.0, device=device)
                mse_obs = 0.0

            loss_cls = F.cross_entropy(logits, c)

            # --- Kalibriere cls_weight auf erstes verfügbares Batch ---
            if (cls_w is None) and calibrate_cls_weight and (not calibrated) and (Tin > 1):
                obs0 = float(loss_obs.detach().item())
                cls0 = float(loss_cls.detach().item())
                # Ziel: cls_w * cls0 ~= cls_target_ratio * obs0
                cls_w = (cls_target_ratio * obs0) / (cls0 + 1e-8)
                cls_w = float(np.clip(cls_w, 1e-6, 10.0))
                calibrated = True
                if verbose:
                    print(f"[calib] cls_weight={cls_w:.6g} (target_ratio={cls_target_ratio}, obs0={obs0:.4g}, cls0={cls0:.4g})")

            if cls_w is None:
                # Fallback: falls Tin<=1 o. ä.
                cls_w = float(cls_target_ratio)

            loss = loss_obs + float(cls_w) * loss_cls

            # Gradient Accumulation für Memory-Fairness
            (loss / max(1, int(grad_accum_steps))).backward()

            do_step = ((step + 1) % max(1, int(grad_accum_steps)) == 0) or (step + 1 == len(train_loader))
            if do_step:
                if max_grad_norm is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), float(max_grad_norm))
                opt.step()
                opt.zero_grad(set_to_none=True)

            tr_losses.append(float(loss.detach().item()))
            tr_mse_obs.append(mse_obs)

            tr_y.extend(c.detach().cpu().tolist())
            tr_p.extend(torch.argmax(logits.detach(), dim=1).cpu().tolist())

        tr_acc = accuracy_score(tr_y, tr_p) if tr_y else 0.0

        # ---------------- val ----------------
        model.eval()
        va_losses = []
        va_y, va_p = [], []
        va_mse_obs = []

        with torch.no_grad():
            for x_in, dt, y_all, c in val_loader:
                x_in = x_in.to(device)
                dt = dt.to(device)
                c = c.to(device)

                Tin = int(x_in.size(1))

                yhat, logits, warm_preds = model(x_in, dt, n_future=0, return_warmup_preds=True)

                if Tin > 1:
                    tgt_obs = x_in[:, 1:Tin, :2]
                    loss_obs = F.mse_loss(warm_preds, tgt_obs)
                    mse_obs = float(loss_obs.detach().item())
                else:
                    loss_obs = torch.tensor(0.0, device=device)
                    mse_obs = 0.0

                loss_cls = F.cross_entropy(logits, c)
                loss = loss_obs + float(cls_w) * loss_cls

                va_losses.append(float(loss.detach().item()))
                va_mse_obs.append(mse_obs)

                va_y.extend(c.detach().cpu().tolist())
                va_p.extend(torch.argmax(logits.detach(), dim=1).cpu().tolist())

        va_acc = accuracy_score(va_y, va_p) if va_y else 0.0

        tr_loss = float(np.mean(tr_losses)) if tr_losses else 0.0
        va_loss = float(np.mean(va_losses)) if va_losses else 0.0

        hist["tr_loss"].append(tr_loss)
        hist["va_loss"].append(va_loss)
        hist["tr_acc"].append(tr_acc)
        hist["va_acc"].append(va_acc)
        hist["tr_mse_obs"].append(float(np.mean(tr_mse_obs)) if tr_mse_obs else 0.0)
        hist["va_mse_obs"].append(float(np.mean(va_mse_obs)) if va_mse_obs else 0.0)
        hist["cls_weight"].append(float(cls_w))

        improved = va_loss < best["loss"] - 1e-10
        if improved:
            best["loss"] = va_loss
            best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose:
            print(
                f"[C] ep={ep:03d} tr_loss={tr_loss:.4f} va_loss={va_loss:.4f} "
                f"| tr_acc={tr_acc:.3f} va_acc={va_acc:.3f} "
                f"| va_mse_obs(1-step)={hist['va_mse_obs'][-1]:.6f} "
                f"| cls_w={float(cls_w):.3g} "
                f"| bad={bad_epochs:02d}/{patience}"
            )

        if patience is not None and bad_epochs >= int(patience):
            if verbose:
                print(f"[early-stop] Stop bei ep={ep} (keine Val-Verbesserung in {patience} Epochen). Best va_loss={best['loss']:.6f}")
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"])
    return hist


def eval_extrap(
    model: ExtrapSeq2SeqModel,
    loader: DataLoader,
    tin: int = 30,
    max_steps: int = 300,
    split_name: str = "test",
    y_ring_m: float = 3.048,
    scaler=None,
):
    scaler = _get_scaler_or_raise(scaler)

    model.eval()
    all_y, all_p = [], []

    se_total = 0.0
    n_total = 0

    se_obs = 0.0
    n_obs = 0

    se_fut = 0.0
    n_fut = 0

    with torch.no_grad():
        for x_in, dt, y_all, c in loader:
            x_in = x_in.to(device)
            dt = dt.to(device)
            y_all = y_all.to(device)
            c = c.to(device)

            Tin = int(x_in.size(1))
            T = int(y_all.size(1))
            n_future = max(0, T - Tin)

            # Klassifikation (nur Beobachtungsfenster)
            _, logits = model(x_in, dt, n_future=0)

            # Regression: volle Länge (Tin + n_future = T)
            if n_future > 0:
                y_hat_full, _, warmup_preds = model(
                    x_in, dt, n_future=n_future, return_warmup_preds=True
                )
                # y_hat_full: (B, Tin+n_future, 2) enthält im Obs-Teil nur Kopie von x_in[:, :, :2]
                # => Für eine "echte" Obs-Prediction nutzen wir die warmup 1‑Step‑Preds.
                obs_pred = torch.cat([x_in[:, 0:1, :2], warmup_preds], dim=1)  # (B,Tin,2)
                fut_pred = y_hat_full[:, Tin:Tin + n_future, :]                # (B,n_future,2)
                y_pred_full = torch.cat([obs_pred, fut_pred], dim=1)           # (B,T,2)
            else:
                # nur Beobachtungsfenster vorhanden
                y_hat_full, _, warmup_preds = model(
                    x_in, dt, n_future=0, return_warmup_preds=True
                )
                obs_pred = torch.cat([x_in[:, 0:1, :2], warmup_preds], dim=1)  # (B,Tin,2)
                y_pred_full = obs_pred

            # --- MSE (skaliertes Koordinatensystem; konsistent zu y_all) ---
            diff = (y_pred_full - y_all)
            se_total += float((diff ** 2).sum().item())
            n_total += int(diff.numel())

            # Beobachtungsfenster
            diff_obs = (y_pred_full[:, :Tin, :] - y_all[:, :Tin, :])
            se_obs += float((diff_obs ** 2).sum().item())
            n_obs += int(diff_obs.numel())

            # Future
            if n_future > 0:
                diff_f = (y_pred_full[:, Tin:, :] - y_all[:, Tin:, :])
                se_fut += float((diff_f ** 2).sum().item())
                n_fut += int(diff_f.numel())

            all_y.extend(c.cpu().tolist())
            all_p.extend(torch.argmax(logits, dim=1).cpu().tolist())

    acc = accuracy_score(all_y, all_p) if all_y else 0.0

    mse_total = float(se_total / max(1, n_total))
    mse_obs_v = float(se_obs / max(1, n_obs))
    mse_fut_v = float(se_fut / max(1, n_fut)) if n_fut > 0 else 0.0

    print(
        f"[{split_name}] acc={acc:.3f} | mse_total={mse_total:.6f} | "
        f"mse_obs={mse_obs_v:.6f} | mse_future={mse_fut_v:.6f}"
    )

    cm = confusion_matrix(all_y, all_p, labels=list(range(3)))
    return {
        "acc": acc,
        "mse_total": mse_total,
        "mse_obs": mse_obs_v,
        "mse_future": mse_fut_v,
        "cm": cm,
        "y_true": all_y,
        "y_pred": all_p,
    }


In [ ]:
def plot_confusion_matrix(
    cm,
    class_names=None,
    normalize: bool = False,
    title: str = "Konfusionsmatrix",
    ax=None,
    cmap: str = "viridis",
    show_colorbar: bool = False,
    show: bool = True,
):
    if isinstance(cm, dict):
        if "cm" in cm:
            cm = cm["cm"]
        elif ("y_true" in cm) and ("y_pred" in cm):
            cm = confusion_matrix(cm["y_true"], cm["y_pred"], labels=[0, 1, 2])
        else:
            raise ValueError(
                "Wenn 'cm' ein dict ist, muss es entweder den Key 'cm' "
                "oder die Keys ('y_true','y_pred') enthalten."
            )

    cm = np.asarray(cm)
    if cm.ndim != 2 or cm.shape[0] != cm.shape[1]:
        raise ValueError(f"cm muss quadratisch sein (N,N). Erhalten: {cm.shape}")

    n = cm.shape[0]

    if class_names is None:
        class_names = [str(i) for i in range(n)]
    class_names = [str(c) for c in class_names]
    if len(class_names) != n:
        raise ValueError(f"class_names muss Länge {n} haben. Erhalten: {len(class_names)}")

    cm_plot = cm.astype(float)

    # Optional normalisieren (pro true-class = Zeile)
    if normalize:
        row_sums = cm_plot.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        cm_plot = cm_plot / row_sums

    created_ax = ax is None
    if ax is None:
        # entspricht ~600x390 px bei dpi=100 (wie im Beispiel)
        fig, ax = plt.subplots(figsize=(6.0, 3.9), dpi=100)
    else:
        fig = ax.figure

    im = ax.imshow(cm_plot, interpolation="nearest", cmap=cmap)

    if show_colorbar:
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.set_ylabel("Anteil" if normalize else "Anzahl", rotation=90)

    ax.set_title(title)
    ax.set_xlabel("Vorhergesagte Klasse")
    ax.set_ylabel("Wahre Klasse")
    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)

    # Zellwerte (wie im Beispiel: schwarze Schrift)
    fmt = ".2f" if normalize else "d"
    for i in range(n):
        for j in range(n):
            if normalize:
                text = format(cm_plot[i, j], fmt)
            else:
                text = format(int(round(cm[i, j])), fmt)

            ax.text(j, i, text, ha="center", va="center", color="black")

    ax.set_aspect("equal")
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(n - 0.5, -0.5)
    fig.tight_layout()

    if show and created_ax:
        plt.show()

    return fig, ax


@torch.no_grad()
def evaluate_seq2seq_classification_metrics(
    model,
    loader,
    class_names=None,
    averages=("macro", "weighted", "micro"),
    zero_division: int = 0,
    include_report: bool = True,
    report_digits: int = 4,
    return_report_dict: bool = True,
    print_report: bool = True,
):
    model.eval()

    y_true, y_pred = [], []
    n_classes = None

    for x_in, dt, _y_all, labels in loader:
        x_in = x_in.to(device)
        dt = dt.to(device)
        labels = labels.to(device)

        _, logits = model(x_in, dt, n_future=0)
        if n_classes is None:
            n_classes = int(logits.size(1))

        pred = torch.argmax(logits, dim=1)
        y_true.extend(labels.detach().cpu().tolist())
        y_pred.extend(pred.detach().cpu().tolist())

    # Labels/Names robust setzen
    if n_classes is None:
        n_classes = len(class_names) if class_names is not None else 3

    label_ids = list(range(n_classes))
    if class_names is None:
        class_names = [str(i) for i in label_ids]

    # Metriken
    acc = accuracy_score(y_true, y_pred) if len(y_true) else 0.0
    metrics = {"accuracy": float(acc)}

    for avg in averages:
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=avg, zero_division=zero_division
        )
        metrics[f"precision_{avg}"] = float(p)
        metrics[f"recall_{avg}"]    = float(r)
        metrics[f"f1_{avg}"]        = float(f1)

    # classification_report
    report = None
    if include_report:
        if return_report_dict:
            report = classification_report(
                y_true,
                y_pred,
                labels=label_ids,
                target_names=class_names,
                digits=report_digits,
                zero_division=zero_division,
                output_dict=True,
            )
        else:
            report = classification_report(
                y_true,
                y_pred,
                labels=label_ids,
                target_names=class_names,
                digits=report_digits,
                zero_division=zero_division,
                output_dict=False,
            )

    # Ausgabe
    print("=== Seq2Seq-LSTM: Klassifikation (ohne Confusion Matrix) ===")
    print(f"Accuracy: {metrics['accuracy']:.6f}")
    for avg in averages:
        print(
            f"{avg:>8s} | "
            f"Precision: {metrics[f'precision_{avg}']:.6f} | "
            f"Recall: {metrics[f'recall_{avg}']:.6f} | "
            f"F1: {metrics[f'f1_{avg}']:.6f}"
        )

    if include_report and print_report:
        print("\n=== classification_report ===")
        # Für konsistente Textausgabe auch bei dict-return:
        print(
            classification_report(
                y_true,
                y_pred,
                labels=label_ids,
                target_names=class_names,
                digits=report_digits,
                zero_division=zero_division,
                output_dict=False,
            )
        )

    return metrics, report

def evaluate_seq2seq_trajectory_mse_vs_clean_gt(
    model,
    loader_obs,
    loader_clean_gt,
    scaler=None,
    return_parts: bool = True,
):
    scaler = _get_scaler_or_raise(scaler)
    model.eval()

    def _ensure_loader(obj, ref_loader: DataLoader | None):
        # Case 1: already a DataLoader
        if isinstance(obj, DataLoader):
            return obj

        # Case 2: dataset -> wrap using ref_loader settings
        if ref_loader is None:
            raise ValueError(
                "Wenn ein Dataset übergeben wird, muss ref_loader (DataLoader) vorhanden sein, "
                "um batch_size und collate_fn zu übernehmen."
            )

        bs = getattr(ref_loader, "batch_size", None)
        cf = getattr(ref_loader, "collate_fn", None)
        if bs is None or cf is None:
            raise ValueError(
                "Konnte batch_size/collate_fn nicht aus ref_loader ableiten. "
                "Bitte DataLoader direkt übergeben oder ref_loader korrekt initialisieren."
            )

        num_workers = getattr(ref_loader, "num_workers", 0)
        pin_memory = getattr(ref_loader, "pin_memory", False)
        persistent_workers = getattr(ref_loader, "persistent_workers", False) if num_workers > 0 else False

        return DataLoader(
            obj,
            batch_size=bs,
            shuffle=False,
            drop_last=False,
            collate_fn=cf,
            num_workers=num_workers,
            pin_memory=pin_memory,
            persistent_workers=persistent_workers,
        )

    # loader_obs kann (optional) ebenfalls ein Dataset sein
    if not isinstance(loader_obs, DataLoader):
        loader_obs = _ensure_loader(loader_obs, ref_loader=_ensure_loader(loader_clean_gt, None))  # selten gebraucht

    loader_clean_gt = _ensure_loader(loader_clean_gt, ref_loader=loader_obs)

    # --- Konsistenzchecks ---
    if len(loader_obs.dataset) != len(loader_clean_gt.dataset):
        raise ValueError(
            "loader_obs und loader_clean_gt müssen gleich viele Samples enthalten "
            f"({len(loader_obs.dataset)} vs {len(loader_clean_gt.dataset)})."
        )
    if getattr(loader_obs, "batch_size", None) != getattr(loader_clean_gt, "batch_size", None):
        raise ValueError(
            "loader_obs und loader_clean_gt sollten dieselbe batch_size haben, "
            "damit zip(...) sauber aligned."
        )

    obs_ids = getattr(loader_obs.dataset, "throw_ids", None)
    gt_ids  = getattr(loader_clean_gt.dataset, "throw_ids", None)
    if obs_ids is not None and gt_ids is not None and list(obs_ids) != list(gt_ids):
        raise ValueError(
            "Die Reihenfolge der throw_ids stimmt zwischen loader_obs und loader_clean_gt nicht überein. "
            "Bitte sicherstellen: gleicher Split, shuffle=False, identische Dataset-Reihenfolge."
        )

    device_ = next(model.parameters()).device

    sse_full = 0.0
    n_full = 0
    sse_obs = 0.0
    n_obs = 0
    sse_future = 0.0
    n_future_count = 0

    for (x_in, dt, _y_any, labels_obs), (_x_gt, _dt_gt, y_clean, labels_gt) in zip(loader_obs, loader_clean_gt):
        x_in = x_in.to(device_)
        dt = dt.to(device_)
        y_clean = y_clean.to(device_)

        # Alignment-Check über Labels (hilfreich, falls Reihenfolge verrutscht)
        if not torch.equal(labels_obs, labels_gt):
            raise ValueError(
                "Mismatch zwischen Labels aus loader_obs und loader_clean_gt innerhalb eines Batches. "
                "Das deutet auf fehlende Alignment/Reihenfolge hin."
            )

        B, Tin, _ = x_in.shape
        T = int(y_clean.size(1))
        n_future = max(0, T - Tin)

        if n_future > 0:
            y_hat_full, _logits, warmup_preds = model(x_in, dt, n_future=n_future, return_warmup_preds=True)
            obs_pred = torch.cat([x_in[:, 0:1, :2], warmup_preds], dim=1)      # (B,Tin,2)
            fut_pred = y_hat_full[:, Tin:Tin + n_future, :]                    # (B,n_future,2)
            y_pred_full = torch.cat([obs_pred, fut_pred], dim=1)               # (B,T,2)
        else:
            _y_hat_full, _logits, warmup_preds = model(x_in, dt, n_future=0, return_warmup_preds=True)
            y_pred_full = torch.cat([x_in[:, 0:1, :2], warmup_preds], dim=1)   # (B,T,2)

        pred_np = y_pred_full.detach().cpu().numpy().reshape(-1, 2)
        gt_np   = y_clean.detach().cpu().numpy().reshape(-1, 2)

        pred_xy = scaler.inverse_transform(pred_np).reshape(B, T, 2)
        gt_xy   = scaler.inverse_transform(gt_np).reshape(B, T, 2)

        diff = pred_xy - gt_xy
        sse_full += float((diff ** 2).sum())
        n_full += int(diff.size)

        if return_parts:
            diff_obs = diff[:, :Tin, :]
            sse_obs += float((diff_obs ** 2).sum())
            n_obs += int(diff_obs.size)

            diff_fut = diff[:, Tin:, :]
            if diff_fut.size > 0:
                sse_future += float((diff_fut ** 2).sum())
                n_future_count += int(diff_fut.size)

    out = {"mse_full_vs_clean_gt": sse_full / max(1, n_full)}
    if return_parts:
        out["mse_obs_vs_clean_gt"] = sse_obs / max(1, n_obs)
        out["mse_future_vs_clean_gt"] = (sse_future / n_future_count) if n_future_count > 0 else float("nan")

    print("=== Seq2Seq-LSTM: Trajektorienfehler vs. clean Ground Truth (originale Einheiten) ===")
    print(f"MSE full:   {out['mse_full_vs_clean_gt']:.8f}")
    if return_parts:
        print(f"MSE obs:    {out['mse_obs_vs_clean_gt']:.8f}")
        print(f"MSE future: {out['mse_future_vs_clean_gt']:.8f}")

    return out


## 7) Modus A (noisy) mit  ExtrapSeq2SeqModel
Encoder bekommt **Tin=50** noisy Punkte

In [ ]:
batch_size = 64  # pro Schritt (anpassbar bei OOM)
effective_batch_size = 64  # Vergleich: identisch zur effektiven Batchsize im PINN
grad_accum_steps = max(1, int(effective_batch_size // batch_size))
assert effective_batch_size % batch_size == 0, "effective_batch_size muss Vielfaches von batch_size sein (für grad accumulation)."


tin = 50

# --- Dynamischer Rollout (rollout_until_ring): keine feste Future-Länge im Inferenz-/Eval-Pfad ---
# max_steps_cap ist nur eine Sicherheits-/Effizienz-Obergrenze (kein Horizontwissen als Feature).
max_steps_cap = 300
print("tin:", tin, "| max_steps_cap:", max_steps_cap)


df_train = df[df["wurf_id"].isin(train_ids)].reset_index(drop=True)
df_val   = df[df["wurf_id"].isin(val_ids)].reset_index(drop=True)
df_test  = df[df["wurf_id"].isin(test_ids)].reset_index(drop=True)

dsA_train = ExtrapolationDataset(df_train, scaler, tin=tin, input_source="noisy", target_source="noisy")
dsA_val   = ExtrapolationDataset(df_val, scaler, tin=tin, input_source="noisy", target_source="noisy")
dsA_test  = ExtrapolationDataset(df_test, scaler, tin=tin, input_source="noisy", target_source="noisy")
dsA_clean_test = ExtrapolationDataset(df_test, scaler, tin=tin, input_source="clean", target_source="clean")

dlA_train = DataLoader(dsA_train, batch_size=batch_size, shuffle=True, drop_last=False, collate_fn=collate_extrap)
dlA_val   = DataLoader(dsA_val,   batch_size=batch_size, shuffle=False, collate_fn=collate_extrap)
dlA_test  = DataLoader(dsA_test,  batch_size=batch_size, shuffle=False, collate_fn=collate_extrap)

modelA = ExtrapSeq2SeqModel(hidden_dim=128, num_layers=2, dropout=0.1).to(device)

histA = train_extrap(
    modelA, dlA_train, dlA_val,
    tin=tin,
    epochs=800, lr=2e-3, cls_weight=None, cls_target_ratio=0.01, calibrate_cls_weight=True, patience=25, grad_accum_steps=grad_accum_steps,
    max_steps_cap=max_steps_cap
)

# plot_history(histA, "Mode A (extrapolation) - ExtrapSeq2SeqModel")

cmA = eval_extrap(modelA, dlA_test, tin=tin,
    max_steps=max_steps_cap, split_name="test A")
plot_confusion_matrix(
    cm=cmA,
    class_names=["0", "1", "2"],
    normalize=False,
    title="",
)
evaluate_seq2seq_classification_metrics(modelA, dlA_test, class_names=["0", "1", "2"])
evaluate_seq2seq_trajectory_mse_vs_clean_gt(
    modelA, dlA_test, dsA_clean_test, scaler=scaler
)

## 8) Modus B (clean-trimmed/extrapolation) mit ExtrapSeq2SeqModel
Encoder bekommt **Tin=30** Punkte, Decoder erzeugt eine **vollständige, komplett vorhergesagte** Trajektorie (T=50 Punkte).

In [ ]:
tin = 30

# --- Dynamischer Rollout (rollout_until_ring): keine feste Future-Länge im Inferenz-/Eval-Pfad ---
max_steps_cap = 300
print("tin:", tin, "| max_steps_cap:", max_steps_cap)

dsB_train = ExtrapolationDataset(df_train, scaler, tin=tin, input_source="clean", target_source="clean")
dsB_val   = ExtrapolationDataset(df_val, scaler, tin=tin, input_source="clean", target_source="clean")
dsB_test  = ExtrapolationDataset(df_test, scaler, tin=tin, input_source="clean", target_source="clean")

dlB_train = DataLoader(dsB_train, batch_size=batch_size, shuffle=True, drop_last=False, collate_fn=collate_extrap)
dlB_val   = DataLoader(dsB_val,   batch_size=batch_size, shuffle=False, collate_fn=collate_extrap)
dlB_test  = DataLoader(dsB_test,  batch_size=batch_size, shuffle=False, collate_fn=collate_extrap)

modelB = ExtrapSeq2SeqModel(hidden_dim=128, num_layers=2, dropout=0.1).to(device)

histB = train_extrap(
    modelB, dlB_train, dlB_val,
    tin=tin,
    epochs=800, lr=2e-3, cls_weight=None, cls_target_ratio=0.01, calibrate_cls_weight=True, patience=25, grad_accum_steps=grad_accum_steps,
    max_steps_cap=max_steps_cap
)

# plot_history(histB, "Mode B (extrapolation) - ExtrapSeq2SeqModel")

cmB = eval_extrap(modelB, dlB_test, tin=tin,
    max_steps=max_steps_cap, split_name="test B")
plot_confusion_matrix(
    cm=cmB,
    class_names=["0", "1", "2"],
    normalize=False,
    title="",
)
evaluate_seq2seq_classification_metrics(modelB, dlB_test, class_names=["0", "1", "2"])
evaluate_seq2seq_trajectory_mse_vs_clean_gt(
    modelB, dlB_test, dsB_test, scaler=scaler
)


## 9) Modus C (noisy-trimmed/extrapolation) mit ExtrapSeq2SeqModel
Encoder bekommt **Tin=30** noisy Punkte, Decoder erzeugt eine **vollständige, komplett vorhergesagte** Trajektorie (T=50 Punkte).

In [ ]:
tin = 30

# --- Dynamischer Rollout (rollout_until_ring): keine feste Future-Länge im Inferenz-/Eval-Pfad ---
max_steps_cap = 300
print("tin:", tin, "| max_steps_cap:", max_steps_cap)

dsC_train = ExtrapolationDataset(df_train, scaler, tin=tin, input_source="noisy", target_source="noisy")
dsC_val   = ExtrapolationDataset(df_val, scaler, tin=tin, input_source="noisy", target_source="noisy")
dsC_test  = ExtrapolationDataset(df_test, scaler, tin=tin, input_source="noisy", target_source="noisy")
dsC_clean_test = ExtrapolationDataset(df_test, scaler, tin=tin, input_source="clean", target_source="clean")

dlC_train = DataLoader(dsC_train, batch_size=batch_size, shuffle=True, drop_last=False, collate_fn=collate_extrap)
dlC_val   = DataLoader(dsC_val,   batch_size=batch_size, shuffle=False, collate_fn=collate_extrap)
dlC_test  = DataLoader(dsC_test,  batch_size=batch_size, shuffle=False, collate_fn=collate_extrap)

modelC = ExtrapSeq2SeqModel(hidden_dim=128, num_layers=2, dropout=0.1).to(device)

histC = train_extrap(
    modelC, dlC_train, dlC_val,
    tin=tin,
    epochs=800, lr=2e-3, cls_weight=None, cls_target_ratio=0.01, calibrate_cls_weight=True, patience=25, grad_accum_steps=grad_accum_steps,
    max_steps_cap=max_steps_cap
)

# plot_history(histC, "Mode C (extrapolation) - ExtrapSeq2SeqModel")

cmC = eval_extrap(modelC, dlC_test, tin=tin,
    max_steps=max_steps_cap, split_name="test C")
plot_confusion_matrix(
    cm=cmC,
    class_names=["0", "1", "2"],
    normalize=False,
    title="",
)
evaluate_seq2seq_classification_metrics(modelC, dlC_test, class_names=["0", "1", "2"])
evaluate_seq2seq_trajectory_mse_vs_clean_gt(
    modelC, dlC_test, dsC_clean_test, scaler=scaler
)


## 10) Qualitative Plots: Trajektorien (Ground Truth vs Prediction)
Wir plotten pro Modus ein Beispiel aus Test.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch


H_RING_M = 3.048  # 10ft in meters (Korbebene)


# --- helper: time normalization exactly like in ExtrapolationDataset ---
def normalize_time_np(t_abs: np.ndarray, t_min: float, denom: float) -> np.ndarray:
    # t_abs: (T,1) float32
    denom = max(1e-8, float(denom))
    return (t_abs - float(t_min)) / denom


def _get_globals_or_raise(model, scaler, device, dg):
    if model is None:
        model = globals().get("model", None)
    if scaler is None:
        scaler = globals().get("scaler", None)
    if device is None:
        device = globals().get("device", None)
    if dg is None:
        dg = globals().get("dg", None)

    missing = [
        name
        for name, obj in [("model", model), ("scaler", scaler), ("device", device)]
        if obj is None
    ]
    if missing:
        raise RuntimeError(
            f"Fehlt im Scope: {missing}. Übergib sie als Parameter oder setze globale Variablen."
        )
    return model, scaler, device, dg


def _ring_crossing_idx_np(y: np.ndarray, y_ring: float = H_RING_M) -> int:
    """Erstes absteigendes Crossing y<=y_ring nach Apex. y: (T,) in Metern."""
    T = len(y)
    apex = int(np.argmax(y))
    for k in range(apex + 1, T):
        if (y[k] <= y_ring) and (y[k - 1] > y_ring):
            return k
    return T - 1


def _build_from_ds_groups(
    ds,
    scaler,
    idx: int,
    n_obs: int | None = None,
    input_source: str | None = None,
    target_source: str | None = None,
):
    """Baut alle Arrays/Tensors wie im Dataset, aber erlaubt n_obs override.

    Gibt zusätzlich unskalierte XYs in Metern fürs Plotten zurück.
    """
    wid = ds.throw_ids[idx]
    g = ds.groups[wid]

    T = len(g)
    Tin = int(n_obs) if n_obs is not None else int(ds.tin)
    assert 1 <= Tin <= T, f"Tin/n_obs={Tin} ist ungültig für T={T}"

    in_src = input_source if input_source is not None else ds.input_source
    tg_src = target_source if target_source is not None else ds.target_source

    # raw (meters)
    t_abs = g["t"].to_numpy(np.float32)[:, None]  # (T,1)
    xy_clean = g[["x_clean", "y_clean"]].to_numpy(np.float32)  # (T,2)
    xy_noisy = g[["x_noisy", "y_noisy"]].to_numpy(np.float32)  # (T,2)

    xy_in_m = xy_noisy if in_src == "noisy" else xy_clean  # for plotting obs
    xy_gt_m = xy_clean if tg_src == "clean" else xy_noisy  # for plotting GT

    # time norm (no future knowledge): denom based on last observed time
    t_min = float(t_abs[0, 0])
    t_obs_max = float(t_abs[Tin - 1, 0])
    denom = max(1e-8, (t_obs_max - t_min))
    t_n = normalize_time_np(t_abs, t_min, denom).astype(np.float32)  # (T,1)

    # Δt ausschließlich aus Beobachtungsfenster
    if Tin >= 2:
        dt_n = float(np.mean(np.diff(t_n[:Tin, 0])))
    else:
        dt_n = 0.0

    # scale XY like training
    xy_in_s = scaler.transform(xy_in_m)  # (T,2)
    xy_gt_s = scaler.transform(xy_gt_m)  # (T,2)

    # model tensors (match ExtrapolationDataset)
    x_in = np.concatenate([xy_in_s[:Tin], t_n[:Tin]], axis=1).astype(np.float32)  # (Tin,3)
    y_all = xy_gt_s.astype(np.float32)  # (T,2)

    # ring height in scaled space
    y_ring_scaled = (H_RING_M - float(scaler.mean_[0, 1])) / float(scaler.std_[0, 1])

    label = int(g["label"].iloc[0])

    return {
        "wurf_id": wid,
        "label": label,
        "Tin": Tin,
        "T": T,
        "t_min": t_min,
        "denom": float(denom),
        "dt_n": dt_n,
        "xy_obs_m": xy_in_m[:Tin],  # (Tin,2) meters
        "xy_gt_m": xy_gt_m,         # (T,2) meters
        "x_in": torch.from_numpy(x_in),          # (Tin,3)
        "dt": torch.tensor([dt_n], dtype=torch.float32),  # (1,)
        "y_all": torch.from_numpy(y_all),        # (T,2)
        "y_ring_scaled": torch.tensor([y_ring_scaled], dtype=torch.float32),  # (1,)
        "input_source": in_src,
        "target_source": tg_src,
    }


@torch.no_grad()
def demo_reconstruction(
    ds,
    idx=0,
    n_obs=None,
    model=None,
    scaler=None,
    device=None,
    dg=None,
    xlim=(-0.1, 5.0),
    ylim=(2.4, 4.15),
    max_steps=300,
):
    """Qualitative Demo (ohne t_all): Rollout dynamisch bis Korbebene."""
    model, scaler, device, dg = _get_globals_or_raise(model, scaler, device, dg)

    s = _build_from_ds_groups(ds, scaler, idx=idx, n_obs=n_obs)
    wid = s["wurf_id"]

    model.eval()

    # dynamischer Rollout bis Ring (nur falls Zukunftsschritte benötigt werden)
    if (max_steps is None) or (int(max_steps) <= 0) or (s["Tin"] >= s["T"]):
        y_future = torch.zeros((1, 0, 2), device=device)
        stop_idx = torch.tensor([-1], device=device)
        S = 0
    else:
        y_future, stop_idx, S = model.rollout_until_ring(
            s["x_in"].unsqueeze(0).to(device),
            s["dt"].unsqueeze(0).to(device),
            s["y_ring_scaled"].unsqueeze(0).to(device),
            max_steps=max_steps,
        )
    y_future = y_future.squeeze(0)  # (S,2)

    # label prediction (aus Encoder-Kontext)
    _, logits = model(
        s["x_in"].unsqueeze(0).to(device),
        s["dt"].unsqueeze(0).to(device),
        n_future=0,
    )
    pred_label = int(torch.argmax(logits, dim=1).item())
    gt_label = int(s["label"])

    # inverse scale preds to meters
    y_future_m = scaler.inverse_transform(y_future.cpu().numpy())  # (S,2)

    # build full pred curve: observed + future
    xy_pred_m = np.vstack([s["xy_obs_m"], y_future_m])

    # GT (wie bisher: bis Ringhöhe für Visualisierung)
    k_stop_gt = _ring_crossing_idx_np(s["xy_gt_m"][:, 1], y_ring=H_RING_M)
    xy_gt_cut = s["xy_gt_m"][: k_stop_gt + 1]

    print(f"Wurf {wid} | true label={gt_label} | pred label={pred_label}")
    print(
        f"Tin={s['Tin']} | dt_n={s['dt_n']:.6f} | input={s['input_source']} | target(GT)={s['target_source']}"
    )

    # --- Plot: identisch zum amortisierten PINN (Default-Farbzyklus, Labels, Limits) ---
    noise_str = "rauschfrei" if s["input_source"] == "clean" else "verrauscht"
    n_obs_plot = int(n_obs) if n_obs is not None else int(s["Tin"])

    plt.figure()
    plt.scatter(
        s["xy_obs_m"][:, 0], s["xy_obs_m"][:, 1], s=20, label="Beobachtungen (Eingabe)"
    )
    plt.plot(xy_gt_cut[:, 0], xy_gt_cut[:, 1], linewidth=2, label="Referenztrajektorie")
    plt.plot(xy_pred_m[:, 0], xy_pred_m[:, 1], linewidth=2, label="Modellvorhersage")
    plt.hlines(
        y=3.048,
        xmin=4.115 - 0.2286,
        xmax=4.115 + 0.2286,
        linewidth=3,
        label="Korb (10ft)",
    )

    # plt.title(
    #     f"Trajektorienvorhersage: Seq2Seq-LSTM, N_obs={n_obs_plot}, {noise_str} |"
    #     f"Wurf-ID={wid} |"
    #     f"(GT={gt_label}, Pred={pred_label})\n"
    # )
    plt.xlabel("x [m]")
    plt.ylabel("y [m]")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.show()


@torch.no_grad()
def demo_reconstruction_noisy_vs_clean(
    ds_noisy,
    ds_clean,
    idx=0,
    n_obs=None,
    model=None,
    scaler=None,
    device=None,
    dg=None,
    xlim=(-0.1, 5.0),
    ylim=(2.4, 4.15),
    max_steps=300,
):
    """Noisy-Obs vs Clean-GT Demo (ohne t_all): Rollout dynamisch bis Korbebene."""
    model, scaler, device, dg = _get_globals_or_raise(model, scaler, device, dg)

    # sample from noisy ds
    sN = _build_from_ds_groups(
        ds_noisy,
        scaler,
        idx=idx,
        n_obs=n_obs,
        input_source="noisy",
        target_source=ds_noisy.target_source,
    )
    wid = sN["wurf_id"]

    # build index cache for ds_clean by wurf_id
    if not hasattr(demo_reconstruction_noisy_vs_clean, "_clean_index") or getattr(
        demo_reconstruction_noisy_vs_clean, "_clean_index_ds_id", None
    ) != id(ds_clean):
        clean_index = {ds_clean.throw_ids[j]: j for j in range(len(ds_clean))}
        demo_reconstruction_noisy_vs_clean._clean_index = clean_index
        demo_reconstruction_noisy_vs_clean._clean_index_ds_id = id(ds_clean)

    clean_index = demo_reconstruction_noisy_vs_clean._clean_index
    if wid not in clean_index:
        raise KeyError(f"wurf_id={wid} nicht im ds_clean gefunden. Prüfe Split/IDs.")

    sC = _build_from_ds_groups(
        ds_clean,
        scaler,
        idx=clean_index[wid],
        n_obs=n_obs,
        input_source="clean",
        target_source="clean",
    )

    # dynamischer Rollout aus noisy Input bis Ring
    if (max_steps is None) or (int(max_steps) <= 0) or (sN["Tin"] >= sN["T"]):
        y_future = torch.zeros((1, 0, 2), device=device)
        stop_idx = torch.tensor([-1], device=device)
        S = 0
    else:
        y_future, stop_idx, S = model.rollout_until_ring(
            sN["x_in"].unsqueeze(0).to(device),
            sN["dt"].unsqueeze(0).to(device),
            sN["y_ring_scaled"].unsqueeze(0).to(device),
            max_steps=max_steps,
        )
    y_future = y_future.squeeze(0)
    y_future_m = scaler.inverse_transform(y_future.cpu().numpy())
    xy_pred_m = np.vstack([sN["xy_obs_m"], y_future_m])

    # GT clean bis Ring
    k_stop_gt = _ring_crossing_idx_np(sC["xy_gt_m"][:, 1], y_ring=H_RING_M)
    xy_gt_cut = sC["xy_gt_m"][: k_stop_gt + 1]

    # class pred
    _, logits = model(
        sN["x_in"].unsqueeze(0).to(device),
        sN["dt"].unsqueeze(0).to(device),
        n_future=0,
    )
    pred_label = int(torch.argmax(logits, dim=1).item())
    gt_label = int(sN["label"])

    print(f"Wurf {wid} | true label={gt_label} | pred label={pred_label}")
    print(f"Tin={sN['Tin']} | dt_n={sN['dt_n']:.6f} | noisy→(pred) | clean GT")

    # --- Plot: identisch zum amortisierten PINN ---
    n_obs_plot = int(n_obs) if n_obs is not None else int(sN["Tin"])

    plt.figure()
    plt.scatter(
        sN["xy_obs_m"][:, 0],
        sN["xy_obs_m"][:, 1],
        s=20,
        label="Beobachtungen (Eingabe)",
    )
    plt.plot(xy_gt_cut[:, 0], xy_gt_cut[:, 1], linewidth=2, label="Referenztrajektorie")
    plt.plot(xy_pred_m[:, 0], xy_pred_m[:, 1], linewidth=2, label="Modellvorhersage")
    plt.hlines(
        y=3.048,
        xmin=4.115 - 0.2286,
        xmax=4.115 + 0.2286,
        linewidth=3,
        label="Korb (10ft)",
    )

    # plt.title(
    #     f"Trajektorienvorhersage: Seq2Seq-LSTM, N_obs={n_obs_plot}, verrauscht |"
    #     f"Wurf-ID={wid} |"
    #     f"(GT={gt_label}, Pred={pred_label})\n"
    # )
    plt.xlabel("x [m]")
    plt.ylabel("y [m]")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.show()


demo_reconstruction_noisy_vs_clean(
    ds_noisy=dsA_test,
    ds_clean=dsA_clean_test,
    idx=14,
    n_obs=50,
    model=modelA,
    scaler=scaler,
    device=device,
)

demo_reconstruction(
    ds=dsB_test,
    idx=0,
    n_obs=30,
    model=modelB,
    scaler=scaler,
    device=device,
)

demo_reconstruction_noisy_vs_clean(
    ds_noisy=dsC_test,
    ds_clean=dsA_clean_test,
    idx=11,
    n_obs=30,
    model=modelC,
    scaler=scaler,
    device=device,
)